# RetailIQ — Exploratory Data Analysis
This notebook explores the cleaned Sample Superstore transactions. Run `python -m src.prepare_data` first.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import kpis, customer_summary
df = pd.read_csv(ROOT / 'data/processed/superstore_cleaned.csv', parse_dates=['order_date','ship_date','month_year'])
df.head()

## Data quality and shape

In [ ]:
print('Shape:', df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Exact duplicates:', df.duplicated().sum())

## Business KPIs
Orders are counted using distinct Order IDs rather than transaction rows.

In [ ]:
kpis(df)

## Monthly revenue and profit

In [ ]:
monthly = df.groupby('month_year', as_index=False).agg(sales=('sales','sum'), profit=('profit','sum'))
ax = monthly.plot(x='month_year', y=['sales','profit'], figsize=(12,5), marker='o', title='Monthly revenue and profit')
ax.set_xlabel('Month'); ax.set_ylabel('Amount'); ax.grid(alpha=.2); plt.show()

## Category and regional performance

In [ ]:
display(df.groupby('category').agg(revenue=('sales','sum'), profit=('profit','sum')).sort_values('revenue', ascending=False))
display(df.groupby('region').agg(revenue=('sales','sum'), profit=('profit','sum')).sort_values('profit', ascending=False))

## Customer repeat-purchase analysis

In [ ]:
customers = customer_summary(df)
display(customers.head(10))
display(customers.groupby('customer_type').agg(customers=('customer_id','count'), revenue=('revenue','sum')))


## Discount and profitability
Correlation is descriptive and should not be interpreted as causal evidence.

In [ ]:
print('Discount-profit correlation:', df[['discount','profit']].corr().iloc[0,1])
display(df.groupby('discount').agg(lines=('order_id','size'), avg_profit=('profit','mean'), total_profit=('profit','sum')).reset_index())